<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/6_Neo4j_Contexto_Relacional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir S6 en Colab"></a>

**Acceso público:** [página del curso](https://jazaineam1.github.io/BigData2026/) · **Laboratorio guiado:** [checklist paso a paso ↗](https://jazaineam1.github.io/BigData2026/assets/tutoriales/s06-laboratorio-guiado.html) · **Tutorial visual Aura:** [Neo4j + Cypher + vista Graph ↗](https://jazaineam1.github.io/BigData2026/assets/tutoriales/neo4j-aura-s06-paso-a-paso.html)


# Sesión 6 — De la fila priorizada al contexto relacional con Neo4j

## Universidad Central
> ### Facultad de Ingeniería y Ciencias Básicas
> ### Maestría en Analítica de Datos — BIG DATA (64491093)

**Caso conductor:** Compras Claras

**Pregunta profesional.** Laura ya sabe qué proceso revisar primero. Antes de asignarlo a un auditor, necesita comprender **qué actores se conectan con ese proceso y por qué caminos**.

**Idea de hoy en una frase:** una tabla puede calcular estas conexiones; un grafo las convierte en parte explícita de la consulta y, además, permite **verlas**.

### Producto observable

Al terminar tendrás una **ficha relacional de revisión** con:

1. el proceso de S5 o un respaldo pedagógico claramente declarado;
2. un modelo `Entidad → Proceso → Proveedor`;
3. un grafo consultado realmente en Neo4j/Aura o marcado como pendiente si usaste respaldo;
4. el contraste H2-R verificado con pandas ↔ Neo4j;
5. un proveedor explorado y su vecindario;
6. una decisión de modelado, una alternativa descartada y un límite analítico;
7. `s06_contexto_procesos.jsonl`, entrada de S7.

> **Regla de interpretación de toda la sesión:** conectividad describe estructura. **No demuestra irregularidad, colusión ni riesgo por sí sola.**

**Entorno:** navegador + Google Colab + Neo4j AuraDB + Cypher.


## El hilo del caso



**Cómo se lee.** Cada sesión entrega el producto que abre la siguiente. S6 no empieza una historia nueva: toma una fila priorizada y pregunta por su contexto relacional.

**Qué nos dice.** El cambio de tecnología responde a un cambio de pregunta.

**Qué NO permite concluir todavía.** Continuar el hilo no convierte las relaciones encontradas en evidencia de conducta irregular.

**Qué error común.** Tratar MongoDB, Cassandra y Neo4j como una lista de tecnologías sin una pregunta que justifique cada una.


## ¿Por qué aparece Neo4j aquí?

**Neo4j es una base de datos de grafos de propiedades.** Representa entidades como nodos, hechos como relaciones y atributos como propiedades. Cypher permite expresar patrones que siguen esas relaciones.

| Modelo | Cómo responde la pregunta | Idea para hoy |
|---|---|---|
| SQL / pandas | cruces y agregaciones | puede calcular la respuesta y será nuestro contrato |
| MongoDB | documentos + `$lookup` | útil cuando el documento es la unidad natural |
| Cassandra | tabla diseñada para una consulta conocida | excelente cuando el patrón de acceso está definido de antemano |
| Neo4j | nodos + relaciones + patrones | hace explícito el recorrido y permite inspeccionarlo visualmente |

**PARA LLEVAR.** No vamos a probar que Neo4j sea “más rápido”. Vamos a comprobar que representa correctamente una pregunta relacional y que la visualización ayuda a comprender caminos.

### La misma idea fuera de contratación

Red profesional: Persona → Empresa → Persona.  
Rutas: Ciudad → Carretera → Ciudad.  
Recomendación: Usuario → Contenido → Usuario.  
Transferencias: Cuenta → Transferencia → Cuenta.


## Mapa de la sesión

| Bloque | Qué haces | Resultado |
|---|---|---|
| 1. Ancla y datos | recuperas el caso sin rehacer S5 | proceso + entidad + historial disponible |
| 2. Modelo mental | dibujas nodos, relaciones, caminos y patrones | sabes leer Cypher |
| 3. Aura + carga | conectas, creas restricciones y cargas el grafo | motor real consultable |
| 4. **Graph Lab** | ejecutas consultas visuales y exploras grados/conexiones | entiendes el grafo con los ojos |
| 5. H2-R | calculas en pandas y reproduces en Neo4j | corrección comprobada |
| 6. Tu vecindario | eliges, filtras y escribes Cypher | decisión propia |
| 7. Hito | exportas evidencia y límites | ficha + JSONL para S7 |

### Semáforo

- 🧠 **ENTIENDE:** debes poder explicarlo.
- ▶️ **EJECUTA:** corre la celda y verifica la salida.
- ✏️ **MODIFICA:** cambia el dato indicado.
- 🌐 **AURA:** ocurre en la interfaz Query de Neo4j, no dentro de Colab.


## Rúbrica S06 — léela antes del laboratorio

| Criterio | Evidencia completa | Peso |
|---|---|---:|
| Continuidad y traza | proceso/NIT, origen del ancla, autor, fecha y commit privado | 15 |
| Modelo | justifica `Proceso` como nodo y compara una alternativa | 20 |
| Ejecución | conexión Neo4j, consulta propia, vecindario y filtro correctos | 20 |
| Verificación | pandas = Neo4j e idempotencia de carga | 15 |
| Decisión y H2-R | máximo, mediana, proveedor H2-R y proveedor explorado diferenciados | 15 |
| Límite | qué no puedes concluir y qué dato faltaría | 15 |

**Total: 100.** Si usas RESPALDO, puedes continuar; la ficha debe dejar Neo4j como **PENDIENTE**, nunca como ejecutado.


In [ ]:
#@title Preparar interactividad { display-mode: "form" }
import base64, json, html as html_lib
from IPython.display import display, HTML

def pregunta_codificada(token):
    p = json.loads(base64.b64decode(token).decode("utf-8"))
    uid = f"s06-p{p['numero']}"
    opts = "".join(
        f'<label style="display:block;margin:8px 0"><input type="radio" name="{uid}" value="{i}"> {html_lib.escape(op)}</label>'
        for i, op in enumerate(p["opciones"])
    )
    handler = (
        "const box=this.closest('[data-pregunta]');"
        "const e=box.querySelector('input:checked');"
        "const s=box.querySelector('[aria-live]');"
        "if(!e){s.textContent='Selecciona una opción.';return;}"
        f"const i=Number(e.value),r={json.dumps(p['retro'], ensure_ascii=False)};"
        f"const ok=i==={p['correcta']};"
        "s.textContent=(ok?'Correcto. ':'Incorrecto. ')+r[i];"
        "s.style.background=ok?'#dcfce7':'#fee2e2';"
        "s.style.color=ok?'#14532d':'#7f1d1d';"
        "s.style.padding='12px';"
        "s.style.marginTop='10px';"
        "s.style.borderRadius='8px';"
    )
    handler = html_lib.escape(handler, quote=True)
    contador = str(p['numero']) + (f" de {p['total']}" if p.get('total') else '')
    box = (
        f'<div data-pregunta="{uid}" style="border:2px solid #1e40af;background:#eff6ff;color:#172554;border-radius:12px;padding:15px;margin:14px 0">'
        f'<strong>Pregunta {contador} — {html_lib.escape(p["tema"])}</strong>'
        f'<p style="background:#fef3c7;color:#713f12;padding:10px;border-radius:8px">{html_lib.escape(p.get("contexto", "Aplica lo que acabas de observar en el caso de Laura."))}</p>'
        f'<p>{html_lib.escape(p["pregunta"])}</p>{opts}'
        f'<button onclick="{handler}" style="background:#1e40af;color:white;border:0;border-radius:7px;padding:8px 12px">Verificar respuesta</button>'
        f'<div id="r-{uid}" aria-live="polite"></div></div>'
    )
    display(HTML(box))

def tutorial(url, alto=760):
    box = f'<iframe src="{url}?embed=1" width="100%" height="{alto}" style="border:0;border-radius:10px;background:#faf7ef"></iframe>'
    box += f'<p><a href="{url}" target="_blank">Abrir tutorial en pantalla completa ↗</a></p>'
    display(HTML(box))

print("Soporte S6 listo.")


---
# 1. Recuperar el proceso y preparar el contexto

S6 funciona de dos maneras:

- **Continuidad real:** si conservaste `s05_ancla_s06.json`, actívalo en la celda de ancla.
- **Modo autónomo:** si no tienes ese archivo, S6 usa una ancla pedagógica versionada y lo declara explícitamente.

Esto corrige una ambigüedad importante: **no llamaremos “mi proceso de S5” a una ancla que en realidad viene del curso.**


In [ ]:
#@title Cargar extracto relacional { display-mode: "form" }
import json
import urllib.request
import hashlib
from pathlib import Path
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv"
MANIFEST_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional_manifest.json"

rutas = []
for nombre, url in [("s06_contexto_relacional.csv", DATA_URL), ("s06_contexto_relacional_manifest.json", MANIFEST_URL)]:
    archivo = Path(nombre)
    if not archivo.is_file() and (Path("Datos") / nombre).is_file():
        archivo = Path("Datos") / nombre
    if not archivo.is_file():
        try:
            with urllib.request.urlopen(url, timeout=30) as respuesta:
                contenido = respuesta.read()
            archivo.write_bytes(contenido)
        except Exception as exc:
            raise RuntimeError("Carga fallida. Sube el CSV y el manifest del curso a Archivos y repite esta celda.") from exc
    rutas.append(archivo)

datos = pd.read_csv(
    rutas[0],
    dtype={"nit_entidad": str, "nit_proveedor": str, "id_proceso": str},
    keep_default_na=False,
)
for columna in ["nit_entidad", "nit_proveedor", "id_proceso"]:
    datos[columna] = datos[columna].str.strip()
for columna in ["precio_base", "valor_adjudicado", "noticias_entidad"]:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")

assert datos["nit_entidad"].ne("").all() and datos["id_proceso"].ne("").all(), "Falta una identidad obligatoria."
manifest = json.loads(rutas[1].read_text(encoding="utf-8-sig"))
huella_datos = hashlib.sha256(rutas[0].read_bytes()).hexdigest()

print("Filas disponibles:", len(datos))
print("Huella SHA256 del extracto:", huella_datos)


### ¿Quieres continuar exactamente desde S5?

Si descargaste `s05_ancla_s06.json` al final de S5:

1. marca `USAR_MI_ANCLA_S5 = True`;
2. ejecuta la celda;
3. si Colab te lo pide, selecciona ese archivo.

Si no lo tienes, deja `False`. **No es un error:** trabajarás con el respaldo pedagógico y la ficha lo registrará.


In [ ]:
#@title Elegir ancla propia S5 o respaldo { display-mode: "form" }
# Si en S5 descargaste s05_ancla_s06.json, puedes usarlo.
# La práctica sigue funcionando de manera autónoma si no lo tienes.
USAR_MI_ANCLA_S5 = False  #@param {type:"boolean"}

archivo_s5 = Path("s05_ancla_s06.json")
if USAR_MI_ANCLA_S5 and not archivo_s5.is_file():
    try:
        from google.colab import files
        subidos = files.upload()
        if "s05_ancla_s06.json" not in subidos:
            raise ValueError("Debes seleccionar exactamente s05_ancla_s06.json.")
    except ImportError:
        raise RuntimeError("Coloca s05_ancla_s06.json en el directorio de trabajo y repite la celda.")

if USAR_MI_ANCLA_S5:
    ancla_original = json.loads(archivo_s5.read_text(encoding="utf-8"))
    requeridos = {"id_proceso", "entidad", "nit_entidad"}
    faltan = requeridos - set(ancla_original)
    if faltan:
        raise ValueError(f"El archivo de S5 no contiene los campos requeridos: {sorted(faltan)}")
    origen_ancla = "archivo propio S5"
else:
    ancla_original = dict(manifest["ancla_pedagogica"])
    origen_ancla = "ancla pedagógica versionada incluida en S6"

print("Origen:", origen_ancla)
print(json.dumps(ancla_original, ensure_ascii=False, indent=2))


### Cómo se lee la entrada

**Cómo se lee.** `id_proceso` identifica la fila de trabajo; `nit_entidad` identifica la entidad para construir su contexto.

**Qué nos dice.** Tenemos un punto de partida concreto.

**Qué NO permite concluir todavía.** Una fila priorizada no es una sentencia de riesgo ni una adjudicación problemática.

**Qué error común.** Inventar un proveedor para el candidato si la fuente no lo reporta.


### Procedencia y unidad de observación

El extracto de S6 conserva dos funciones diferentes:

- **candidato S5:** ancla la pregunta;
- **registro adjudicado del extracto:** aporta relaciones observadas hacia proveedores.

La selección del curso parte de `1.000 → 163 → 77` candidatos y amplía el contexto con procesos adjudicados de las entidades y proveedores compartidos.

> **Importante:** “registro adjudicado del extracto” no significa automáticamente “ocurrió antes que el candidato”. La fecha de publicación debe revisarse si la pregunta exige precedencia temporal.

| Campo | Uso |
|---|---|
| `id_proceso` | identidad del nodo Proceso |
| `nit_entidad` | identidad analítica del nodo Entidad |
| `nit_proveedor` | identidad analítica del nodo Proveedor |
| `precio_base` | presupuesto base reportado |
| `valor_adjudicado` | valor adjudicado reportado |
| `fecha_publicacion` | contexto temporal |
| `descripcion`, `url_secop` | texto y trazabilidad para S7 |


In [ ]:
fechas = pd.to_datetime(datos["fecha_publicacion"], errors="coerce", utc=True, format="mixed")
print("Fechas de publicación válidas:", int(fechas.notna().sum()), "de", len(datos))
print("Intervalo observado:", fechas.min(), "a", fechas.max())


In [ ]:
nit_deseado = str(ancla_original["nit_entidad"]).strip()
hist = datos[datos["tipo_registro"].eq("historico_adjudicado")].copy()
assert hist["nit_proveedor"].ne("").all(), "Un registro histórico carece de NIT de proveedor."

nombres_proveedor = hist.groupby("nit_proveedor")["proveedor"].agg(
    lambda nombres: " / ".join(sorted(set(nombres)))
)
variantes = hist.groupby("nit_proveedor")["proveedor"].nunique()
print("NIT de proveedor con varios nombres:", int(variantes.gt(1).sum()))

hist_ancla = hist[hist["nit_entidad"].eq(nit_deseado)]
if hist_ancla.empty:
    print("Tu entidad no tiene historial en este extracto. El trabajo continúa con el respaldo pedagógico declarado.")
    ancla_trabajo = dict(manifest["ancla_pedagogica"])
    nit_deseado = str(ancla_trabajo["nit_entidad"]).strip()
    hist_ancla = hist[hist["nit_entidad"].eq(nit_deseado)]
    uso_respaldo_s06 = True
else:
    ancla_trabajo = ancla_original
    uso_respaldo_s06 = origen_ancla != "archivo propio S5"

print("Entidad de trabajo:", ancla_trabajo["entidad"])
print("Procesos adjudicados del extracto:", hist_ancla["id_proceso"].nunique())
print("Proveedores distintos:", hist_ancla["nit_proveedor"].nunique())
print("H2-R evaluable con mi propia ancla:", not uso_respaldo_s06)


**Cómo se lee.** Los conteos anteriores describen adjudicaciones disponibles para la entidad de trabajo, no el candidato aislado.

**Qué nos dice.** Hay suficiente material para construir relaciones `Entidad → Proceso → Proveedor`.

**Qué NO permite concluir todavía.** Más contratos o más proveedores no equivalen a más riesgo.

**Qué error común.** Confundir “historial disponible” con “todo el universo contractual” o con “información disponible antes del candidato”.


---
# 2. Antes de Cypher: aprende a ver el grafo

## Ejemplo manual mínimo



En el dibujo, **Proveedor X es un puente** porque aparece conectado a procesos de dos entidades distintas.

No existe una relación directa “Entidad A conoce a Entidad B”. La conexión emerge por el camino:

`Entidad A → Proceso P-101 → Proveedor X ← Proceso P-202 ← Entidad B`

Ese cambio de perspectiva —de filas a caminos— es la idea central de la sesión.


In [ ]:
#@title Autoevaluación 1 — Evidencia de una relación { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAxLCAidGVtYSI6ICJFdmlkZW5jaWEgZGUgdW5hIHJlbGFjacOzbiIsICJwcmVndW50YSI6ICLCv0N1w6FuZG8gcG9kZW1vcyBjcmVhciBgQURKVURJQ0FET19BYCBlbnRyZSB1biBwcm9jZXNvIHkgdW4gcHJvdmVlZG9yPyIsICJvcGNpb25lcyI6IFsiQ3VhbmRvIGVsIHByb3ZlZWRvciBhcGFyZWNlIGVuIHVuYSBub3RpY2lhIGRlIGxhIGVudGlkYWQuIiwgIkN1YW5kbyBleGlzdGUgdW4gcmVnaXN0cm8gYWRqdWRpY2FkbyBjb24gaWRlbnRpZmljYWRvciBkZSBwcm92ZWVkb3IgaW5mb3JtYWRvLiIsICJDdWFuZG8gZWwgcHJvY2VzbyB0aWVuZSB1biBwcmVjaW8gYmFzZSBhbHRvLiIsICJDdWFuZG8gcXVlcmVtb3MgY29tcGxldGFyIHZpc3VhbG1lbnRlIHVuIGNhbWluby4iXSwgImNvcnJlY3RhIjogMSwgInJldHJvIjogWyJVbmEgbm90aWNpYSBlcyBjb250ZXh0bzsgbm8gcHJ1ZWJhIGxhIGFkanVkaWNhY2nDs24gY29uY3JldGEuIiwgIkNvcnJlY3RvOiBsYSByZWxhY2nDs24gcmVwcmVzZW50YSB1biBoZWNobyBvYnNlcnZhZG8gZW4gbGEgZnVlbnRlLiIsICJFbCBwcmVzdXB1ZXN0byBubyBpZGVudGlmaWNhIGFsIGFkanVkaWNhdGFyaW8uIiwgIlVuIGdyYWZvIG5vIGRlYmUgY29tcGxldGFyIGhlY2hvcyBhdXNlbnRlcyBwYXJhIHF1ZSBzZSB2ZWEgbWVqb3IuIl0sICJjb250ZXh0byI6ICJFbCBjYW5kaWRhdG8gZGUgUzUgcHVlZGUgbm8gdHJhZXIgcHJvdmVlZG9yLiBMb3MgcmVnaXN0cm9zIGFkanVkaWNhZG9zIHPDrSBwdWVkZW4gYXBvcnRhcmxvLiIsICJ0b3RhbCI6IDEwfQ==")


## Seis conceptos y nada más

| Concepto | Qué significa | Ejemplo |
|---|---|---|
| **Nodo** | una entidad con identidad | un proveedor |
| **Label** | categoría del nodo | `Proveedor` |
| **Propiedad** | dato guardado | `nit: "900..."` |
| **Relación** | hecho dirigido entre nodos | `ADJUDICADO_A` |
| **Camino** | secuencia conectada | Entidad → Proceso → Proveedor |
| **Patrón** | la forma que pides encontrar | `(e)-[:PUBLICA]->(p)` |

### Lee Cypher como una frase

`(e:Entidad {nit:"123"})`

- `e`: nombre temporal de la variable;
- `Entidad`: label;
- `{nit:"123"}`: propiedad usada para identificarla.

`(e:Entidad)-[:PUBLICA]->(p:Proceso)`

se lee: **“una Entidad conectada mediante PUBLICA con un Proceso”.**

`MATCH` significa: **busca coincidencias con esta forma.**


In [ ]:
#@title Autoevaluación 2 — Patrón { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAyLCAidGVtYSI6ICJQYXRyw7NuIiwgInByZWd1bnRhIjogIsK/UXXDqSBleHByZXNhIGBNQVRDSCAoZTpFbnRpZGFkKS1bOlBVQkxJQ0FdLT4ocDpQcm9jZXNvKWA/IiwgIm9wY2lvbmVzIjogWyJDcmVhciBzaWVtcHJlIHVuYSBlbnRpZGFkIHkgdW4gcHJvY2VzbyBudWV2b3MuIiwgIkJ1c2NhciBjb2luY2lkZW5jaWFzIGRvbmRlIHVuYSBFbnRpZGFkIHB1YmxpY2EgdW4gUHJvY2Vzby4iLCAiQ29udmVydGlyIGxhIHJlbGFjacOzbiBQVUJMSUNBIGVuIHVuYSB0YWJsYSBwZXJtYW5lbnRlLiIsICJPcmRlbmFyIHRvZG9zIGxvcyBwcm9jZXNvcyBwb3IgZW50aWRhZC4iXSwgImNvcnJlY3RhIjogMSwgInJldHJvIjogWyJgTUFUQ0hgIGJ1c2NhOyBubyBjcmVhLiIsICJDb3JyZWN0bzogZWwgcGF0csOzbiBlcyBsYSBmb3JtYSBxdWUgc2UgaW50ZW50YSBlbmNvbnRyYXIuIiwgImBNQVRDSGAgbm8gY3JlYSB1bmEgdGFibGEgcGVybWFuZW50ZS4iLCAiRWwgcGF0csOzbiBubyBpbmNsdXllIGBPUkRFUiBCWWAuIl0sICJjb250ZXh0byI6ICJFbiBDeXBoZXIsIGxhIGZvcm1hIGRlIGxhIGNvbnN1bHRhIHRhbWJpw6luIGNvbnRpZW5lIGxhcyByZWxhY2lvbmVzIHF1ZSBxdWllcmVzIHJlY29ycmVyLiIsICJ0b3RhbCI6IDEwfQ==")


### EJERCICIO S06-PATRON — ahora sí hay un hueco

Completa únicamente el nombre de la relación entre un proceso adjudicado y su proveedor.

**Meta:** obtener `(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)`.


In [ ]:
RELACION_PROCESO_PROVEEDOR = "____"
if RELACION_PROCESO_PROVEEDOR != "ADJUDICADO_A":
    raise ValueError("Revisa el modelo: la relación observada entre Proceso y Proveedor se llama ADJUDICADO_A.")
patron_estudiante = f"(p:Proceso)-[:{RELACION_PROCESO_PROVEEDOR}]->(v:Proveedor)"
print(patron_estudiante)
print("Patrón correcto.")


## Modelo mínimo de S6



`(e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)`

### ¿Por qué `Proceso` es nodo?

Porque tiene identidad propia (`id_proceso`), propiedades propias y participa en otros caminos. Además, S7 reutilizará su texto.

### ¿Por qué Entidad y Proveedor son labels separados?

Es una **decisión simplificadora por rol**: facilita leer el laboratorio.

Una alternativa más general sería usar un nodo `Organizacion {nit}` y representar los roles mediante labels o relaciones. Esa alternativa evita duplicar conceptualmente una organización que pudiera aparecer en más de un rol, pero añade complejidad innecesaria para el objetivo de esta sesión.

### Las relaciones también pueden tener propiedades

Hoy guardaremos `valor_adjudicado` sobre `ADJUDICADO_A`. Eso refuerza la idea de **grafo de propiedades**: no solo los nodos pueden almacenar datos.


## Cypher mínimo

| Construcción | Intuición |
|---|---|
| `MATCH` | busca un patrón |
| `WHERE` | filtra coincidencias |
| `WITH` | pasa variables/resultados a la siguiente etapa |
| `RETURN` | define qué ves |
| `MERGE` | encuentra o crea por una identidad |
| `SET` | actualiza propiedades |
| `UNWIND` | convierte una lista en filas de trabajo |
| `DISTINCT` | evita contar varias veces la misma entidad/nodo |
| `ORDER BY` | ordena |
| `LIMIT` | acota la salida |

### Una regla que evita muchos errores

Usa en `MERGE` la **identidad estable** (`nit`, `id`) y deja propiedades cambiantes para `SET`.

### ¿Dónde se ejecuta cada cosa?

- Celda Python → **Colab**.
- String Cypher enviado por `driver.execute_query(...)` → lo ejecuta Neo4j, aunque lo lances desde **Colab**.
- Consulta que quieres **ver como grafo** → pégala en **Aura → Query** y usa la vista **Graph**.


In [ ]:
#@title Autoevaluación 3 — Identidad { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAzLCAidGVtYSI6ICJJZGVudGlkYWQiLCAicHJlZ3VudGEiOiAiRG9zIGZpbGFzIHRpZW5lbiBlbCBtaXNtbyBOSVQgZGUgcHJvdmVlZG9yIHBlcm8gbm9tYnJlcyBlc2NyaXRvcyBkZSBmb3JtYSBkaWZlcmVudGUuIMK/Q8OzbW8gbW9kZWxhbW9zIGxhIGlkZW50aWRhZD8iLCAib3BjaW9uZXMiOiBbIlVuIHNvbG8gbm9kbyBwb3IgTklUIHkgY29uc2VydmFtb3MgdmFyaWFudGVzIGRlIG5vbWJyZS4iLCAiVW4gbm9kbyBkaWZlcmVudGUgcG9yIGNhZGEgZXNjcml0dXJhIGRlbCBub21icmUuIiwgIkVsaW1pbmFtb3MgdW5hIGZpbGEgcGFyYSBldml0YXIgZHVwbGljYWRvcy4iLCAiVXNhbW9zIG5vbWJyZSArIE5JVCBjb21vIGlkZW50aWRhZCBvYmxpZ2F0b3JpYS4iXSwgImNvcnJlY3RhIjogMCwgInJldHJvIjogWyJDb3JyZWN0bzogZWwgTklUIHJlcG9ydGFkbyBlcyBsYSBpZGVudGlkYWQgYW5hbMOtdGljYSBkZWwgbGFib3JhdG9yaW8uIiwgIkVzbyBmcmFnbWVudGFyw61hIGVsIG1pc21vIGlkZW50aWZpY2Fkb3IuIiwgIlJlcGV0aXIgTklUIGVuIGRpZmVyZW50ZXMgcHJvY2Vzb3MgcHVlZGUgc2VyIHVuIGhlY2hvIHbDoWxpZG8uIiwgIkFncmVnYXIgZWwgbm9tYnJlIGEgbGEgY2xhdmUgcHVlZGUgc2VwYXJhciB2YXJpYW50ZXMgZGVsIG1pc21vIGFjdG9yLiJdLCAiY29udGV4dG8iOiAiTGEgaWRlbnRpZGFkIGRlbCBsYWJvcmF0b3JpbyBzZSBiYXNhIGVuIGVsIE5JVCByZXBvcnRhZG87IGVzbyBubyBzdXN0aXR1eWUgdmVyaWZpY2FjacOzbiBqdXLDrWRpY2EuIiwgInRvdGFsIjogMTB9")


---
# 3. AuraDB: del modelo al motor real

Antes de cargar datos debes poder distinguir cuatro cosas:

- **Neo4j:** motor de base de datos de grafos.
- **AuraDB:** servicio administrado donde ejecutaremos Neo4j.
- **Cypher:** lenguaje de consulta.
- **Driver Neo4j:** librería usada por Python/Colab para conectarse.

### Ruta mínima si el tutorial embebido no carga

1. Abre Aura Console.
2. Entra a **AuraDB** y crea/reutiliza una instancia de práctica.
3. Guarda `Connection URI`, usuario y contraseña.
4. Abre **Query**.
5. Ejecuta `RETURN 1 AS conexion`.
6. Regresa a Colab y conecta con el driver.

El tutorial completo además enseña la vista **Graph**, zoom, propiedades, expansión y las consultas WOW.


In [ ]:
#@title Abrir tutorial visual Neo4j Aura { display-mode: "form" }
tutorial("https://jazaineam1.github.io/BigData2026/assets/tutoriales/neo4j-aura-s06-paso-a-paso.html", alto=820)


In [ ]:
#@title Conectar Aura o activar respaldo { display-mode: "form" }
modo = input("Enter = Aura; escribe RESPALDO si no puedes usar el servicio: ").strip().upper()
if modo not in ["", "RESPALDO"]:
    raise ValueError("Usa Enter o RESPALDO.")
modo_neo4j = modo != "RESPALDO"

if globals().get("driver") is not None:
    driver.close()
driver = None

if modo_neo4j:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neo4j>=6,<7"])
    from getpass import getpass
    from neo4j import GraphDatabase

    URI = input("Connection URI: ").strip()
    USER = input("User name: ").strip()
    PASSWORD = getpass("Password (no se muestra): ")
    if not URI or not USER or not PASSWORD:
        raise ValueError("URI, usuario y contraseña son obligatorios.")

    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
    del PASSWORD
    try:
        driver.verify_connectivity()
    except Exception:
        driver.close()
        driver = None
        raise RuntimeError("Conexión fallida. Revisa URI/usuario/contraseña o repite la celda y usa RESPALDO.") from None
    print("Conexión Neo4j verificada.")
else:
    print("RESPALDO pandas: Neo4j, Graph view, CRUD y comparación quedan PENDIENTES.")


## Identidad e idempotencia antes de cargar

Las restricciones no son decoración: protegen las identidades que usarás en `MERGE`.

- `Entidad.nit` debe ser único.
- `Proceso.id` debe ser único.
- `Proveedor.nit` debe ser único.

Después repetiremos la carga para comprobar que los conteos de nodos y relaciones no crecen.


In [ ]:
if modo_neo4j:
    constraints = [
        "CREATE CONSTRAINT entidad_nit IF NOT EXISTS FOR (e:Entidad) REQUIRE e.nit IS UNIQUE",
        "CREATE CONSTRAINT proceso_id IF NOT EXISTS FOR (p:Proceso) REQUIRE p.id IS UNIQUE",
        "CREATE CONSTRAINT proveedor_nit IF NOT EXISTS FOR (v:Proveedor) REQUIRE v.nit IS UNIQUE",
    ]
    for q in constraints:
        driver.execute_query(q)
    print("Restricciones listas.")
else:
    print("Restricciones Neo4j: PENDIENTES (respaldo).")


### `UNWIND` en una frase

`UNWIND $filas AS fila` toma una lista enviada por Python y permite procesar **cada elemento como una fila dentro de la consulta**.

En esta práctica cargamos alrededor de dos mil registros. Eso es adecuado para el laboratorio.

> **Producción ≠ laboratorio.** Para volúmenes masivos se evalúan lotes, transacciones, índices, mecanismos de importación y planes de consulta. No extrapoles esta celda a “así se cargan millones de registros”.


In [ ]:
#@title Autoevaluación 4 — Idempotencia { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA0LCAidGVtYSI6ICJJZGVtcG90ZW5jaWEiLCAicHJlZ3VudGEiOiAiwr9RdcOpIGNvbWJpbmFjacOzbiBwcm90ZWdlIG1lam9yIGxhIGlkZW50aWRhZCBjdWFuZG8gbGEgbWlzbWEgY2FyZ2Egc2UgZWplY3V0YSBkb3MgdmVjZXM/IiwgIm9wY2lvbmVzIjogWyJgQ1JFQVRFYCBwYXJhIHRvZG9zIGxvcyBub2Rvcy4iLCAiYE1FUkdFYCBwb3IgSUQgZXN0YWJsZSArIHJlc3RyaWNjaW9uZXMgZGUgdW5pY2lkYWQgKyBgU0VUYCBwYXJhIHByb3BpZWRhZGVzLiIsICJgTElNSVQgMWAgZGVzcHXDqXMgZGUgY3JlYXIuIiwgIlVzYXIgZWwgbm9tYnJlIGNvbW8gw7puaWNhIGlkZW50aWRhZC4iXSwgImNvcnJlY3RhIjogMSwgInJldHJvIjogWyJgQ1JFQVRFYCBwdWVkZSBkdXBsaWNhciBlbnRpZGFkZXMgbMOzZ2ljYXMuIiwgIkNvcnJlY3RvOiBpZGVudGlkYWQgZXN0YWJsZSwgcmVzdHJpY2Npw7NuIHkgYWN0dWFsaXphY2nDs24gc2VwYXJhZGEuIiwgImBMSU1JVGAgbGltaXRhIHJlc3VsdGFkb3MsIG5vIHJldmllcnRlIGVzY3JpdHVyYXMuIiwgIkVsIG5vbWJyZSBwdWVkZSB2YXJpYXIuIl0sICJjb250ZXh0byI6ICJDb2xhYiBwdWVkZSByZWluaWNpYXJzZSB5IGVsIGVzdHVkaWFudGUgcHVlZGUgdm9sdmVyIGEgZWplY3V0YXIgbGEgbWlzbWEgY2FyZ2EuIiwgInRvdGFsIjogMTB9")


In [ ]:
cols = [
    "entidad", "nit_entidad", "departamento_entidad", "id_proceso", "referencia",
    "nombre_proceso", "descripcion", "precio_base", "valor_adjudicado", "modalidad", "proveedor",
    "nit_proveedor", "departamento_proveedor", "noticias_entidad", "nivel_menciones",
    "tipo_registro", "url_secop", "es_proceso_candidato_s05", "es_entidad_candidata_s05",
]

para_carga = datos[cols].copy()
para_carga["proveedor"] = para_carga["nit_proveedor"].map(nombres_proveedor)
nombres_entidad = datos.groupby("nit_entidad")["entidad"].agg(lambda nombres: " / ".join(sorted(set(nombres))))
para_carga["entidad"] = para_carga["nit_entidad"].map(nombres_entidad)

rows = para_carga.astype(object).where(pd.notna(para_carga), None).to_dict("records")
rows_proveedor = [
    r for r in rows
    if r["tipo_registro"] == "historico_adjudicado" and r["nit_proveedor"] not in [None, ""]
]
assert len(rows_proveedor) == len(hist)
assert all(isinstance(r["nit_proveedor"], str) for r in rows_proveedor)

print("Carga preparada:", len(rows), "filas;", len(rows_proveedor), "adjudicaciones válidas.")

query_base = """
UNWIND $filas AS fila
MERGE (e:Entidad {nit: toString(fila.nit_entidad)})
SET e.nombre = fila.entidad,
    e.departamento = fila.departamento_entidad,
    e.es_candidata_s05 = fila.es_entidad_candidata_s05,
    e.noticias_entidad = fila.noticias_entidad,
    e.nivel_menciones = fila.nivel_menciones

MERGE (p:Proceso {id: fila.id_proceso})
SET p.referencia = fila.referencia,
    p.nombre = fila.nombre_proceso,
    p.descripcion = fila.descripcion,
    p.precio_base = fila.precio_base,
    p.modalidad = fila.modalidad,
    p.url = fila.url_secop,
    p.es_candidato_s05 = fila.es_proceso_candidato_s05

MERGE (e)-[:PUBLICA]->(p)
"""

query_proveedor = """
UNWIND $filas AS fila
MATCH (p:Proceso {id: fila.id_proceso})
MERGE (v:Proveedor {nit: toString(fila.nit_proveedor)})
SET v.nombre = fila.proveedor,
    v.departamento = fila.departamento_proveedor
MERGE (p)-[a:ADJUDICADO_A]->(v)
SET a.valor_adjudicado = fila.valor_adjudicado
"""

if modo_neo4j:
    driver.execute_query(query_base, filas=rows)
    driver.execute_query(query_proveedor, filas=rows_proveedor)
    print("Carga lista.")
else:
    print("Carga Neo4j: PENDIENTE (respaldo).")


In [ ]:
if modo_neo4j:
    consulta_tamano = """
    MATCH (n) WHERE n:Entidad OR n:Proceso OR n:Proveedor
    WITH count(n) AS nodos
    MATCH ()-[r:PUBLICA|ADJUDICADO_A]->()
    RETURN nodos, count(r) AS relaciones
    """
    antes_carga = driver.execute_query(consulta_tamano).records[0].data()
    driver.execute_query(query_base, filas=rows)
    driver.execute_query(query_proveedor, filas=rows_proveedor)
    despues_carga = driver.execute_query(consulta_tamano).records[0].data()
    carga_repetida = antes_carga == despues_carga
    assert carga_repetida, "La carga repetida alteró los conteos. Revisa IDs y restricciones."
    print("Carga repetida sin crecimiento:", carga_repetida, "|", despues_carga)
else:
    carga_repetida = None
    print("Idempotencia en Neo4j: PENDIENTE.")


**Cómo se lee.** Repetimos exactamente la carga y comparamos conteos.

**Qué nos dice.** Si no crecen, `MERGE` + restricciones preservaron la identidad de los nodos/relaciones en esta ejecución.

**Qué NO permite concluir todavía.** Igualdad de conteos no demuestra que cada propiedad sea correcta ni que la instancia estuviera limpia antes.

**Qué error común.** Llamar `valor` a cualquier cantidad. En el grafo guardamos `p.precio_base` y `a.valor_adjudicado` con nombres distintos.


### RECUPERACIÓN S06 — si Colab se reinició

Ejecuta la celda siguiente. Recupera interactividad, datos y contexto con la ancla pedagógica. Luego vuelve a **Conectar Aura** y repite la carga.

Si estabas trabajando con tu archivo propio de S5, vuelve después a la celda “Elegir ancla propia S5 o respaldo”.


In [ ]:
#@title Recuperar estado S6 { display-mode: "form" }
import base64, json, html as html_lib
from IPython.display import display, HTML

def pregunta_codificada(token):
    p = json.loads(base64.b64decode(token).decode("utf-8"))
    uid = f"s06-p{p['numero']}"
    opts = "".join(
        f'<label style="display:block;margin:8px 0"><input type="radio" name="{uid}" value="{i}"> {html_lib.escape(op)}</label>'
        for i, op in enumerate(p["opciones"])
    )
    handler = (
        "const box=this.closest('[data-pregunta]');"
        "const e=box.querySelector('input:checked');"
        "const s=box.querySelector('[aria-live]');"
        "if(!e){s.textContent='Selecciona una opción.';return;}"
        f"const i=Number(e.value),r={json.dumps(p['retro'], ensure_ascii=False)};"
        f"const ok=i==={p['correcta']};"
        "s.textContent=(ok?'Correcto. ':'Incorrecto. ')+r[i];"
        "s.style.background=ok?'#dcfce7':'#fee2e2';"
        "s.style.color=ok?'#14532d':'#7f1d1d';"
        "s.style.padding='12px';"
        "s.style.marginTop='10px';"
        "s.style.borderRadius='8px';"
    )
    handler = html_lib.escape(handler, quote=True)
    contador = str(p['numero']) + (f" de {p['total']}" if p.get('total') else '')
    box = (
        f'<div data-pregunta="{uid}" style="border:2px solid #1e40af;background:#eff6ff;color:#172554;border-radius:12px;padding:15px;margin:14px 0">'
        f'<strong>Pregunta {contador} — {html_lib.escape(p["tema"])}</strong>'
        f'<p style="background:#fef3c7;color:#713f12;padding:10px;border-radius:8px">{html_lib.escape(p.get("contexto", "Aplica lo que acabas de observar en el caso de Laura."))}</p>'
        f'<p>{html_lib.escape(p["pregunta"])}</p>{opts}'
        f'<button onclick="{handler}" style="background:#1e40af;color:white;border:0;border-radius:7px;padding:8px 12px">Verificar respuesta</button>'
        f'<div id="r-{uid}" aria-live="polite"></div></div>'
    )
    display(HTML(box))

def tutorial(url, alto=760):
    box = f'<iframe src="{url}?embed=1" width="100%" height="{alto}" style="border:0;border-radius:10px;background:#faf7ef"></iframe>'
    box += f'<p><a href="{url}" target="_blank">Abrir tutorial en pantalla completa ↗</a></p>'
    display(HTML(box))

print("Soporte S6 listo.")

import json
import urllib.request
import hashlib
from pathlib import Path
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional.csv"
MANIFEST_URL = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/s06_contexto_relacional_manifest.json"

rutas = []
for nombre, url in [("s06_contexto_relacional.csv", DATA_URL), ("s06_contexto_relacional_manifest.json", MANIFEST_URL)]:
    archivo = Path(nombre)
    if not archivo.is_file() and (Path("Datos") / nombre).is_file():
        archivo = Path("Datos") / nombre
    if not archivo.is_file():
        try:
            with urllib.request.urlopen(url, timeout=30) as respuesta:
                contenido = respuesta.read()
            archivo.write_bytes(contenido)
        except Exception as exc:
            raise RuntimeError("Carga fallida. Sube el CSV y el manifest del curso a Archivos y repite esta celda.") from exc
    rutas.append(archivo)

datos = pd.read_csv(
    rutas[0],
    dtype={"nit_entidad": str, "nit_proveedor": str, "id_proceso": str},
    keep_default_na=False,
)
for columna in ["nit_entidad", "nit_proveedor", "id_proceso"]:
    datos[columna] = datos[columna].str.strip()
for columna in ["precio_base", "valor_adjudicado", "noticias_entidad"]:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")

assert datos["nit_entidad"].ne("").all() and datos["id_proceso"].ne("").all(), "Falta una identidad obligatoria."
manifest = json.loads(rutas[1].read_text(encoding="utf-8-sig"))
huella_datos = hashlib.sha256(rutas[0].read_bytes()).hexdigest()

print("Filas disponibles:", len(datos))
print("Huella SHA256 del extracto:", huella_datos)

USAR_MI_ANCLA_S5 = False
ancla_original = dict(manifest["ancla_pedagogica"])
origen_ancla = "ancla pedagógica versionada incluida en S6"
print("RECUPERACIÓN: se restauró el respaldo pedagógico. Si necesitas tu ancla propia, vuelve a la celda de ancla.")

nit_deseado = str(ancla_original["nit_entidad"]).strip()
hist = datos[datos["tipo_registro"].eq("historico_adjudicado")].copy()
assert hist["nit_proveedor"].ne("").all(), "Un registro histórico carece de NIT de proveedor."

nombres_proveedor = hist.groupby("nit_proveedor")["proveedor"].agg(
    lambda nombres: " / ".join(sorted(set(nombres)))
)
variantes = hist.groupby("nit_proveedor")["proveedor"].nunique()
print("NIT de proveedor con varios nombres:", int(variantes.gt(1).sum()))

hist_ancla = hist[hist["nit_entidad"].eq(nit_deseado)]
if hist_ancla.empty:
    print("Tu entidad no tiene historial en este extracto. El trabajo continúa con el respaldo pedagógico declarado.")
    ancla_trabajo = dict(manifest["ancla_pedagogica"])
    nit_deseado = str(ancla_trabajo["nit_entidad"]).strip()
    hist_ancla = hist[hist["nit_entidad"].eq(nit_deseado)]
    uso_respaldo_s06 = True
else:
    ancla_trabajo = ancla_original
    uso_respaldo_s06 = origen_ancla != "archivo propio S5"

print("Entidad de trabajo:", ancla_trabajo["entidad"])
print("Procesos adjudicados del extracto:", hist_ancla["id_proceso"].nunique())
print("Proveedores distintos:", hist_ancla["nit_proveedor"].nunique())
print("H2-R evaluable con mi propia ancla:", not uso_respaldo_s06)


---
# 4. Graph Lab — ahora sí: mirar, tocar y entender el grafo

Hasta aquí construimos el modelo. En este bloque el objetivo cambia:

> **No quiero solo una tabla correcta; quiero poder explicar visualmente por qué dos entidades están conectadas.**

En Aura → **Query**, una consulta que devuelve nodos, relaciones o caminos puede verse en **Graph**. Una consulta que devuelve solo números se verá principalmente como tabla.

Vamos de cero relaciones a un vecindario real.


In [ ]:
if modo_neo4j:
    r0 = driver.execute_query("MATCH (e:Entidad) RETURN e LIMIT 5")
    print("0 relaciones — nodos Entidad:")
    print(pd.DataFrame([{"nit": r["e"]["nit"], "nombre": r["e"].get("nombre")} for r in r0.records]))

    r1 = driver.execute_query("""
    MATCH (e:Entidad)-[:PUBLICA]->(p:Proceso)
    RETURN e.nombre AS entidad, p.id AS proceso
    LIMIT 5
    """)
    print("\n1 relación — Entidad PUBLICA Proceso:")
    print(pd.DataFrame([r.data() for r in r1.records]))

    r2 = driver.execute_query("""
    MATCH (e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
    RETURN e.nombre AS entidad, p.id AS proceso, v.nombre AS proveedor
    LIMIT 5
    """)
    print("\n2 relaciones — llegamos al proveedor:")
    print(pd.DataFrame([r.data() for r in r2.records]))
else:
    print("Graph Lab en Aura: PENDIENTE. Continúa leyendo las consultas y el respaldo visual.")


## Graph / Table / RAW — aprende qué estás mirando

En **Query**:

- **Graph** muestra nodos y relaciones cuando la consulta los devuelve;
- **Table** facilita comprobar valores y conteos;
- **RAW** muestra la representación técnica de la respuesta.

### Regla práctica

Esto produce una tabla:

```cypher
MATCH (v:Proveedor)
RETURN v.nit, count(*) AS n
```

Esto produce un grafo:

```cypher
MATCH (e:Entidad)-[r:PUBLICA]->(p:Proceso)
RETURN e, r, p
LIMIT 20
```

Y esto devuelve un **camino completo**:

```cypher
MATCH camino=(e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
RETURN camino
LIMIT 20
```

La vista Graph no “adivina” relaciones: **solo dibuja lo que la consulta devuelve**.


In [ ]:
consulta_graph_basico = """
MATCH (e:Entidad)-[pub:PUBLICA]->(p:Proceso)-[adj:ADJUDICADO_A]->(v:Proveedor)
RETURN e, pub, p, adj, v
LIMIT 20
"""
print(consulta_graph_basico)


### 🌐 HAZ ESTO EN AURA → Query

1. Copia la consulta impresa arriba.
2. Pégala en el editor de **Query**.
3. Ejecútala.
4. Selecciona la vista **Graph** si no quedó activa.
5. Haz **Fit to screen** para encajar el resultado.
6. Arrastra un nodo `Proveedor` y separa visualmente sus conexiones.
7. Haz clic en un nodo: revisa sus propiedades.
8. Haz clic en una relación `ADJUDICADO_A`: observa que puede tener `valor_adjudicado`.
9. Haz zoom.
10. Compara Graph con Table antes de continuar.

> Si no ves círculos/flechas y solo números, revisa el `RETURN`: probablemente devolviste escalares en vez de nodos/relaciones/caminos.


In [ ]:
#@title Autoevaluación 5 — Vista Graph { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA1LCAidGVtYSI6ICJWaXN0YSBHcmFwaCIsICJwcmVndW50YSI6ICLCv1F1w6kgZGViZXMgZGV2b2x2ZXIgcGFyYSBxdWUgUXVlcnkgcHVlZGEgZGlidWphciBjbGFyYW1lbnRlIHVuYSByZWxhY2nDs24/IiwgIm9wY2lvbmVzIjogWyJTb2xvIGBjb3VudCgqKWAuIiwgIlNvbG8gZWwgTklUIGNvbW8gdGV4dG8uIiwgIk5vZG9zL3JlbGFjaW9uZXMgbyB1biBjYW1pbm8sIHBvciBlamVtcGxvIGBSRVRVUk4gZSwgciwgcGAgbyBgUkVUVVJOIGNhbWlub2AuIiwgIlVuIERhdGFGcmFtZSBkZSBwYW5kYXMuIl0sICJjb3JyZWN0YSI6IDIsICJyZXRybyI6IFsiVW4gY29udGVvIGVzIGVzY2FsYXI7IHNpcnZlIHBhcmEgVGFibGUsIG5vIHBhcmEgZGlidWphciBlbCBjYW1pbm8uIiwgIlVuIHRleHRvIG5vIGNvbnRpZW5lIGxhIGVzdHJ1Y3R1cmEgZGVsIGdyYWZvLiIsICJDb3JyZWN0bzogbGEgdmlzdWFsaXphY2nDs24gbmVjZXNpdGEgZWxlbWVudG9zIGVzdHJ1Y3R1cmFsZXMgZGVsIGdyYWZvLiIsICJwYW5kYXMgbm8gZXMgbGEgdmlzdGEgR3JhcGggZGUgQXVyYS4iXSwgImNvbnRleHRvIjogIkVsIG1pc21vIE1BVENIIHB1ZWRlIHByb2R1Y2lyIHVuYSB0YWJsYSBvIHVuIGdyYWZvIHNlZ8O6biBsbyBxdWUgZGV2dWVsdmEuIiwgInRvdGFsIjogMTB9")


## Grado: una palabra que conviene usar con precisión

En teoría de grafos, el **grado de un nodo** es el número de relaciones incidentes.

Para un `Proveedor` de nuestro modelo:

- **grado de adjudicaciones:** cuántas relaciones `ADJUDICADO_A` llegan al proveedor;
- **entidades_conectadas:** cuántas `Entidad` distintas aparecen **a dos saltos** del proveedor.

No son lo mismo.

Un proveedor puede tener muchas adjudicaciones con una sola entidad: grado alto, pero pocas entidades distintas.

Esta distinción evita llamar “grado” a cualquier conteo relacional.


In [ ]:
grado_rel = hist.groupby("nit_proveedor")["id_proceso"].nunique().rename("grado_adjudicaciones")
entidades_2saltos = hist.groupby("nit_proveedor")["nit_entidad"].nunique().rename("entidades_conectadas")
grado_df = pd.concat([grado_rel, entidades_2saltos], axis=1).reset_index()
grado_df["proveedor"] = grado_df["nit_proveedor"].map(nombres_proveedor)
grado_df = grado_df.sort_values(
    ["entidades_conectadas", "grado_adjudicaciones", "nit_proveedor"],
    ascending=[False, False, True],
).head(10).reset_index(drop=True)
grado_df


## Efecto WOW — busca el hub y conviértelo en una estrella visual



Lo que quieres reconocer en Aura es esta forma:

- un **Proveedor** en el centro;
- varios **Proceso** alrededor;
- varias **Entidad** detrás de esos procesos;
- caminos repetidos que comparten el mismo proveedor.

La estrella no prueba algo irregular. **Sí revela una estructura que una tabla obliga a reconstruir mentalmente.**


In [ ]:
consulta_wow_global = """
MATCH (v:Proveedor)<-[:ADJUDICADO_A]-(p:Proceso)<-[:PUBLICA]-(e:Entidad)
WITH v, count(DISTINCT e) AS entidades, count(DISTINCT p) AS grado_adjudicaciones
ORDER BY entidades DESC, grado_adjudicaciones DESC, v.nit ASC
LIMIT 1

MATCH camino=(e:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v)
RETURN camino
LIMIT 40
"""
print(consulta_wow_global)


### 🌐 WOW 1 — ejecútala en Aura Query

Cuando aparezca el grafo:

**Primero mira la forma, todavía no los nombres.**

1. Identifica el nodo central `Proveedor`.
2. Cuenta visualmente cuántos brazos aparecen.
3. Observa que cada brazo pasa por `Proceso`.
4. Busca si una misma `Entidad` aparece por varios procesos.
5. Haz clic en el proveedor: revisa `nit` y `nombre`.
6. Haz clic en dos procesos distintos: compara `id`, `modalidad` y `precio_base`.
7. Haz clic en `ADJUDICADO_A`: revisa `valor_adjudicado` cuando esté informado.
8. Arrastra el proveedor al centro y separa los nodos para reducir cruces.
9. Usa **Fit to screen** si perdiste el grafo.
10. Cambia a **Table** para comprobar que la forma visual corresponde a registros reales.

### Cómo evitar engañarte con el dibujo

La posición en pantalla **no es una coordenada analítica**. El layout intenta hacer legible la red. Un nodo “cerca” de otro en píxeles no implica mayor relación estadística.


In [ ]:
#@title Autoevaluación 6 — Grado y conectividad { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA2LCAidGVtYSI6ICJHcmFkbyB5IGNvbmVjdGl2aWRhZCIsICJwcmVndW50YSI6ICJVbiBwcm92ZWVkb3IgdGllbmUgMzAgcHJvY2Vzb3MgYWRqdWRpY2Fkb3MgcGVybyB0b2RvcyBwZXJ0ZW5lY2VuIGEgMiBlbnRpZGFkZXMuIMK/Q3XDoWwgYWZpcm1hY2nDs24gZXMgY29ycmVjdGE/IiwgIm9wY2lvbmVzIjogWyJTdSBncmFkbyBkZSBhZGp1ZGljYWNpb25lcyBwdWVkZSBzZXIgYWx0bywgcGVybyBgZW50aWRhZGVzX2NvbmVjdGFkYXNgIGVzIDIuIiwgIlRpZW5lIG5lY2VzYXJpYW1lbnRlIDMwIGVudGlkYWRlcyBjb25lY3RhZGFzLiIsICJFbCBncmFmbyBkZW11ZXN0cmEgY29vcmRpbmFjacOzbiBlbnRyZSBsYXMgMiBlbnRpZGFkZXMuIiwgIkVsIGxheW91dCB2aXN1YWwgZGViZSBjb2xvY2FybG8gZXhhY3RhbWVudGUgMzAgdmVjZXMgbcOhcyBjZXJjYSBkZWwgY2VudHJvLiJdLCAiY29ycmVjdGEiOiAwLCAicmV0cm8iOiBbIkNvcnJlY3RvOiBlbCBncmFkbyBwb3IgYWRqdWRpY2FjaW9uZXMgeSBsYXMgZW50aWRhZGVzIGRpc3RpbnRhcyBtaWRlbiBjb3NhcyBkaWZlcmVudGVzLiIsICJQcm9jZXNvcyB5IGVudGlkYWRlcyBubyBzb24gbGEgbWlzbWEgdW5pZGFkLiIsICJMYSBlc3RydWN0dXJhIGNvbXBhcnRpZGEgbm8gcHJ1ZWJhIGNvb3JkaW5hY2nDs24uIiwgIkxhcyBkaXN0YW5jaWFzIGRlbCBsYXlvdXQgbm8gc29uIHVuYSBlc2NhbGEgZXN0YWTDrXN0aWNhLiJdLCAiY29udGV4dG8iOiAiUXVlcmVtb3Mgc2VwYXJhciBsYSBtw6l0cmljYSBlc3RydWN0dXJhbCBkZSByZWxhY2lvbmVzIGRpcmVjdGFzIGRlIGxhIG3DqXRyaWNhIGRlIG5lZ29jaW8gYSBkb3Mgc2FsdG9zLiIsICJ0b3RhbCI6IDEwfQ==")


### Si quieres explorar con el mouse

En la vista Graph de Query puedes seleccionar nodos/relaciones para ver propiedades; según la interfaz disponible, el menú contextual permite expandir vecinos o mostrar relaciones. También puedes arrastrar nodos y hacer zoom.

**Hazlo con criterio:** expandir sin límite puede convertir una red legible en una “bola de pelo”. Para aprender, empieza con 10–40 caminos y amplía solo el nodo que estés investigando.

### Caption útil

Si Query muestra un identificador poco legible como caption, usa las opciones de estilo de la vista para que:

- `Entidad` muestre `nombre`;
- `Proceso` muestre `id` o `nombre`;
- `Proveedor` muestre `nombre`.

El NIT sigue siendo la identidad aunque el nombre sea el caption visual.


---
# 5. Ahora sí: H2-R y el contrato pandas

Ya sabes qué significa un camino. Ahora añadimos la comparación analítica.

> **H2-R.** El proveedor adjudicado más conectado de la entidad de trabajo tiene más entidades conectadas que la mediana de los máximos de conectividad de las entidades candidatas S5 con historial.

Es una **comparación descriptiva y falsable sobre este extracto**, no una prueba inferencial ni un modelo de riesgo.


In [ ]:
# Contrato completo: el top 10 se usa solo para presentar; H2-R se calcula sobre todos los proveedores del ancla.
prov_ancla = hist_ancla.groupby("nit_proveedor")["id_proceso"].nunique().rename("procesos_con_entidad")
prov_global = hist.groupby("nit_proveedor")["nit_entidad"].nunique().rename("entidades_conectadas")

resultado_completo_pd = (
    pd.concat([prov_ancla, prov_global], axis=1)
      .loc[prov_ancla.index]
      .reset_index()
)
resultado_completo_pd["procesos_con_entidad"] = resultado_completo_pd["procesos_con_entidad"].astype(int)
resultado_completo_pd["entidades_conectadas"] = resultado_completo_pd["entidades_conectadas"].astype(int)
resultado_completo_pd["proveedor"] = resultado_completo_pd["nit_proveedor"].map(nombres_proveedor)
resultado_completo_pd = resultado_completo_pd.sort_values(
    ["entidades_conectadas", "procesos_con_entidad", "nit_proveedor"],
    ascending=[False, False, True],
).reset_index(drop=True)

esperado_pd = resultado_completo_pd.head(10).copy()

candidatas_hist = hist[hist["es_entidad_candidata_s05"]].copy()
candidatas_hist["conexiones_proveedor"] = candidatas_hist["nit_proveedor"].map(prov_global)
maximos_candidatas = candidatas_hist.groupby("nit_entidad")["conexiones_proveedor"].max()
MEDIANA_H2R = float(maximos_candidatas.median())

if "mediana_maximo_conectadas_por_nit" in manifest:
    assert MEDIANA_H2R == float(manifest["mediana_maximo_conectadas_por_nit"]), "La referencia por NIT no coincide con el extracto."

proveedor_h2r = resultado_completo_pd.iloc[0].copy()
maximo_h2r = int(proveedor_h2r["entidades_conectadas"])

if uso_respaldo_s06:
    desenlace_h2r_pd = "no evaluable con mi ancla; se muestra el respaldo pedagógico"
elif maximo_h2r > MEDIANA_H2R:
    desenlace_h2r_pd = "conexión más fuerte que la mediana de las candidatas de S5"
else:
    desenlace_h2r_pd = "conexión igual o menor que la mediana de las candidatas de S5"

print("Entidades de referencia:", len(maximos_candidatas))
print("Mediana de referencia (candidatas S5):", MEDIANA_H2R)
print("Proveedor que determina H2-R:", proveedor_h2r["nit_proveedor"], "| máximo:", maximo_h2r)
print("Desenlace H2-R (pandas):", desenlace_h2r_pd)
if uso_respaldo_s06:
    print("Comparación ilustrativa del respaldo:", maximo_h2r, ">", MEDIANA_H2R, "=", maximo_h2r > MEDIANA_H2R)

esperado_pd


### Interpretación del contrato

**Cómo se lee.** `procesos_con_entidad` cuenta procesos adjudicados de la entidad ancla. `entidades_conectadas` cuenta entidades distintas asociadas al mismo proveedor.

**Qué nos dice.** Neo4j deberá reproducir esta respuesta.

**Qué NO permite concluir todavía.** Superar la mediana no significa favorecimiento, colusión ni fraude.

**Qué error común.** Confundir el proveedor que determina el máximo de H2-R con el proveedor que después eliges explorar.

**Detalle importante.** H2-R se calcula sobre el resultado completo; `head(10)` existe solo para presentar una tabla manejable.


In [ ]:
#@title Autoevaluación 7 — Mediana H2-R { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA3LCAidGVtYSI6ICJNZWRpYW5hIEgyLVIiLCAicHJlZ3VudGEiOiAiwr9RdcOpIHJlcHJlc2VudGEgYE1FRElBTkFfSDJSYD8iLCAib3BjaW9uZXMiOiBbIkxhIG1lZGlhbmEgZGVsIG7Dum1lcm8gZGUgY29udHJhdG9zIGRlIHRvZG9zIGxvcyBwcm92ZWVkb3JlcyBkZSBTRUNPUC4iLCAiRWwgdW1icmFsIHF1ZSBkZW11ZXN0cmEgcmllc2dvLiIsICJFbCBwdW50byBjZW50cmFsIGRlIGxvcyBtw6F4aW1vcyBkZSBlbnRpZGFkZXMgY29uZWN0YWRhcyBwb3IgY2FkYSBlbnRpZGFkIGNhbmRpZGF0YSBTNSBjb24gaGlzdG9yaWFsLiIsICJFbCBwcm9tZWRpbyBkZWwgcHJvdmVlZG9yIGVsZWdpZG8gcGFyYSBleHBsb3Jhci4iXSwgImNvcnJlY3RhIjogMiwgInJldHJvIjogWyJIMi1SIG5vIHJlc3VtZSB0b2RvcyBsb3MgY29udHJhdG9zIGRlIFNFQ09QLiIsICJFcyB1bmEgcmVmZXJlbmNpYSBkZXNjcmlwdGl2YSwgbm8gdW4gdW1icmFsIHByb2JhZG8gZGUgcmllc2dvLiIsICJDb3JyZWN0bzogcHJpbWVybyBoYXkgdW4gbcOheGltbyBwb3IgZW50aWRhZCB5IGx1ZWdvIHVuYSBtZWRpYW5hIGVudHJlIGVudGlkYWRlcy4iLCAiRWwgcHJvdmVlZG9yIGV4cGxvcmFkbyBwdWVkZSBzZXIgZGlmZXJlbnRlIGRlbCBxdWUgZGV0ZXJtaW5hIEgyLVIuIl0sICJjb250ZXh0byI6ICJMYSByZWZlcmVuY2lhIHNlIGNvbnN0cnV5ZSBjb24gbGFzIGVudGlkYWRlcyBjYW5kaWRhdGFzIFM1IHF1ZSB0aWVuZW4gaGlzdG9yaWFsIGVuIGVsIGV4dHJhY3RvLiIsICJ0b3RhbCI6IDEwfQ==")


In [ ]:
query_contexto = """
MATCH (e:Entidad {nit:$nit})-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
WITH v, count(DISTINCT p) AS procesos_con_entidad
MATCH (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v)
RETURN v.nit AS nit_proveedor,
       v.nombre AS proveedor,
       procesos_con_entidad,
       count(DISTINCT otra) AS entidades_conectadas
ORDER BY entidades_conectadas DESC, procesos_con_entidad DESC, nit_proveedor ASC
LIMIT 10
"""

if modo_neo4j:
    neo = driver.execute_query(query_contexto, nit=nit_deseado)
    neo_df = pd.DataFrame(
        [r.data() for r in neo.records],
        columns=["nit_proveedor", "proveedor", "procesos_con_entidad", "entidades_conectadas"],
    )
    resultado_contexto = neo_df.copy()
else:
    neo_df = None
    resultado_contexto = esperado_pd.copy()
    print("Resultado pandas; consulta Neo4j PENDIENTE.")

resultado_contexto


In [ ]:
coinciden = None
if modo_neo4j:
    cols_cmp = ["nit_proveedor", "procesos_con_entidad", "entidades_conectadas"]
    pd_cmp = esperado_pd[cols_cmp].copy()
    neo_cmp = neo_df[cols_cmp].copy()
    for tabla in [pd_cmp, neo_cmp]:
        tabla["nit_proveedor"] = tabla["nit_proveedor"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
        for columna in cols_cmp[1:]:
            tabla[columna] = tabla[columna].astype("int64")

    coinciden = pd_cmp.reset_index(drop=True).equals(neo_cmp.reset_index(drop=True))
    print("pandas == Neo4j:", coinciden)
    if not coinciden:
        print("\nEsperado pandas:")
        print(pd_cmp.reset_index(drop=True))
        print("\nObtenido Neo4j:")
        print(neo_cmp.reset_index(drop=True))
    assert coinciden, "No sigas: revisa NIT, carga y datos previos de la instancia."
else:
    print("pandas == Neo4j: PENDIENTE; solo se ejecutó pandas.")


**Cómo se lee.** `True` significa que ambas implementaciones producen la misma tabla para las claves y métricas comparadas.

**Qué nos dice.** La consulta del grafo preserva el contrato de resultado.

**Qué NO permite concluir todavía.** No es un benchmark de velocidad y tampoco valida conducta.

**Qué error común.** “Neo4j coincide con pandas, por tanto Neo4j es mejor”. No: solo comprobamos corrección.


In [ ]:
#@title Autoevaluación 8 — Corrección y límites { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA4LCAidGVtYSI6ICJDb3JyZWNjacOzbiB5IGzDrW1pdGVzIiwgInByZWd1bnRhIjogIlNpIGBwYW5kYXMgPT0gTmVvNGpgIGRhIGBUcnVlYCwgwr9xdcOpIHF1ZWTDsyBjb21wcm9iYWRvPyIsICJvcGNpb25lcyI6IFsiUXVlIE5lbzRqIGVzIG3DoXMgcsOhcGlkby4iLCAiUXVlIGV4aXN0ZSBpcnJlZ3VsYXJpZGFkLiIsICJRdWUgdG9kbyBTRUNPUCBlc3TDoSByZXByZXNlbnRhZG8uIiwgIlF1ZSBhbWJhcyBpbXBsZW1lbnRhY2lvbmVzIHByb2R1Y2VuIGVzYSBtaXNtYSByZXNwdWVzdGEgc29icmUgZXN0ZSBleHRyYWN0byB5IGNvbnRyYXRvLiJdLCAiY29ycmVjdGEiOiAzLCAicmV0cm8iOiBbIk5vIHNlIG1pZGnDsyBsYXRlbmNpYSBjb21wYXJhYmxlLiIsICJMYSBpZ3VhbGRhZCBkZSBjb25zdWx0YXMgbm8gZXZhbMO6YSBjb25kdWN0YS4iLCAiRWwgZXh0cmFjdG8gZXMgc2VsZWNjaW9uYWRvLCBubyB1bml2ZXJzYWwuIiwgIkNvcnJlY3RvOiBzZSB2ZXJpZmljw7MgbGEgcmVzcHVlc3RhIGRlIGVzdGEgY29uc3VsdGEgc29icmUgZXN0b3MgZGF0b3MuIl0sICJjb250ZXh0byI6ICJFbCBjb250cmF0byBwYW5kYXMgc2lydmUgY29tbyBvcsOhY3VsbyBkZSBjb3JyZWNjacOzbiwgbm8gY29tbyBiZW5jaG1hcmsgbmkgZGV0ZWN0b3IgZGUgZnJhdWRlLiIsICJ0b3RhbCI6IDEwfQ==")


In [ ]:
nit_literal = str(nit_deseado).replace('"', '\"')
consulta_wow_ancla = f"""
MATCH (ancla:Entidad {{nit:"{nit_literal}"}})-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v:Proveedor)
WITH ancla, v, count(*) AS procesos_ancla
ORDER BY procesos_ancla DESC
LIMIT 1
MATCH camino=(ancla)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v)
             <-[:ADJUDICADO_A]-(:Proceso)<-[:PUBLICA]-(otra:Entidad)
WHERE otra.nit <> ancla.nit
RETURN camino
LIMIT 30
"""
print(consulta_wow_ancla)


### 🌐 WOW 2 — el camino completo Entidad → Proveedor → otra Entidad

Pega la consulta anterior en Aura Query y usa **Graph**.

Esta vez responde:

> “Para mi entidad de trabajo, ¿cómo se ve el proveedor más repetido cuando sigo los caminos hacia otras entidades?”

### Guía de lectura

- Empieza por `ancla:Entidad`.
- Sigue `PUBLICA` hasta `Proceso`.
- Sigue `ADJUDICADO_A` hasta `Proveedor`.
- Desde el proveedor, lee las flechas en sentido inverso hacia otros procesos y entidades.
- Si una entidad aparece varias veces, comprueba si son procesos distintos.
- Haz clic en nodos y relaciones antes de inferir nada por la forma.

**Pregunta para el salón:** ¿qué nodo funciona como puente y qué dato adicional necesitarías antes de decir que la conexión es inusual?


---
# 6. CRUD seguro y tu propio vecindario

Ya viste una red llamativa. Ahora toca trabajar con tu resultado.

Primero demostramos CRUD sobre nodos `S06-DEMO` y los borramos. Después eliges uno de los cinco proveedores de tu resultado y exploras su vecindario.

El proveedor elegido **no tiene que ser** el que determina H2-R. Esa diferencia se registra en la ficha.


In [ ]:
if modo_neo4j:
    driver.execute_query("""
    MERGE (e:Entidad {nit:'S06-E'}) SET e.nombre='Entidad demo'
    MERGE (p:Proceso {id:'S06-DEMO'}) SET p.nombre='Proceso demo'
    MERGE (v:Proveedor {nit:'S06-V'}) SET v.nombre='Proveedor demo'
    MERGE (e)-[:PUBLICA]->(p)
    MERGE (p)-[:ADJUDICADO_A]->(v)
    """)
    r = driver.execute_query(
        "MATCH (p:Proceso {id:'S06-DEMO'}) SET p.estado_revision='revisado' RETURN p.estado_revision AS estado"
    )
    assert r.records[0]["estado"] == "revisado"
    driver.execute_query("MATCH (n) WHERE n.nit IN ['S06-E','S06-V'] OR n.id='S06-DEMO' DETACH DELETE n")
    print("CRUD demo completado y limpiado.")
else:
    print("CRUD Neo4j: PENDIENTE (respaldo).")


In [ ]:
if resultado_contexto.empty:
    raise ValueError("No hay proveedores para elegir.")

opciones = resultado_contexto.head(5).reset_index(drop=True)
for i, row in opciones.iterrows():
    print(f"{i+1:>2}. {row['proveedor']} | entidades={row['entidades_conectadas']} | procesos_ancla={row['procesos_con_entidad']}")

entrada = input("Número de proveedor: ").strip()
try:
    sel = int(entrada)
except ValueError:
    raise ValueError("Escribe un número entero de las opciones mostradas.") from None
if not 1 <= sel <= len(opciones):
    raise ValueError(f"El número debe estar entre 1 y {len(opciones)}.")

proveedor_elegido = opciones.iloc[sel-1]

if modo_neo4j:
    vec = driver.execute_query("""
    MATCH (e:Entidad)-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {nit:$nit})
    RETURN e.nit AS nit_entidad,
           e.nombre AS entidad,
           p.id AS proceso,
           p.nombre AS nombre_proceso,
           p.precio_base AS precio_base
    ORDER BY entidad, precio_base DESC
    """, nit=str(proveedor_elegido["nit_proveedor"]))
    vecindario_df = pd.DataFrame([r.data() for r in vec.records])
else:
    vecindario_df = (
        hist[hist["nit_proveedor"].eq(str(proveedor_elegido["nit_proveedor"]))][
            ["nit_entidad", "entidad", "id_proceso", "nombre_proceso", "precio_base"]
        ]
        .drop_duplicates("id_proceso")
        .rename(columns={"id_proceso": "proceso"})
        .sort_values(["entidad", "precio_base"], ascending=[True, False])
        .reset_index(drop=True)
    )
    print("Vecindario calculado en pandas; ejecución Neo4j PENDIENTE.")

print("Procesos con mi entidad:", int(proveedor_elegido["procesos_con_entidad"]))
print("Entidades conectadas:", int(proveedor_elegido["entidades_conectadas"]))
print("Procesos visibles en el vecindario:", len(vecindario_df))
vecindario_df


### Interpreta tu vecindario

Cada fila representa un proceso conectado al proveedor elegido. Una entidad puede aparecer varias veces porque publicó varios procesos.

`precio_base` sigue siendo presupuesto base, no gasto.

> **No conviertas `entidades_conectadas` en un score de riesgo.** Para hablar de algo inusual necesitarías, como mínimo, temporalidad comparable, mercado/objeto comparable, condiciones de competencia y posiblemente información de propiedad/representación.


### EJERCICIO S06-EXCLUIR

`entidades_conectadas` incluye a la entidad ancla. Completa el operador “distinto de” para contar solo **otras entidades**.

Antes de ejecutar, predice mentalmente: `entidades_conectadas - 1`.


In [ ]:
OPERADOR_EXCLUSION = "____"
if OPERADOR_EXCLUSION != "<>":
    raise ValueError("Necesitamos excluir la entidad ancla. En Cypher, el operador distinto de es <>.")

consulta_otras = f"""
MATCH (otra:Entidad)-[:PUBLICA]->(:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {{nit:$proveedor}})
WHERE otra.nit {OPERADOR_EXCLUSION} $ancla
RETURN count(DISTINCT otra) AS otras_entidades
"""

if modo_neo4j:
    otras_entidades = int(
        driver.execute_query(
            consulta_otras,
            proveedor=str(proveedor_elegido["nit_proveedor"]),
            ancla=nit_deseado,
        ).records[0]["otras_entidades"]
    )
else:
    otras_entidades = int(
        vecindario_df.loc[vecindario_df["nit_entidad"].ne(nit_deseado), "nit_entidad"].nunique()
    )

esperadas_otras = int(proveedor_elegido["entidades_conectadas"]) - 1
assert otras_entidades == esperadas_otras
print("Otras entidades:", otras_entidades, "| esperado:", esperadas_otras)

razon_exploracion = input("Con tus números: ¿por qué explorar este proveedor y cuál de la lista descartaste?: ").strip()
if len(razon_exploracion) < 25:
    raise ValueError("Nombra tu elección, otra opción descartada y al menos un dato de tu salida.")


In [ ]:
#@title Autoevaluación 9 — Proveedor explorado vs H2-R { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiA5LCAidGVtYSI6ICJQcm92ZWVkb3IgZXhwbG9yYWRvIHZzIEgyLVIiLCAicHJlZ3VudGEiOiAiwr9RdcOpIGRlYmUgcmVnaXN0cmFyIGxhIGZpY2hhIHNpIGVsIHByb3ZlZWRvciBlbGVnaWRvIHBhcmEgZXhwbG9yYXIgbm8gZXMgZWwgbWlzbW8gcXVlIGRldGVybWluYSBIMi1SPyIsICJvcGNpb25lcyI6IFsiU29sbyBlbCBwcm92ZWVkb3IgbcOhcyB2aXN0b3NvIGRlbCBncmFmby4iLCAiU29sbyBlbCBwcm92ZWVkb3IgZXhwbG9yYWRvLiIsICJBbWJvcyBOSVQgeSBzdXMgbcOpdHJpY2FzLCBwb3JxdWUgcmVzcG9uZGVuIHByZWd1bnRhcyBkaWZlcmVudGVzLiIsICJEZWJlbW9zIGZvcnphciBxdWUgc2VhbiBlbCBtaXNtby4iXSwgImNvcnJlY3RhIjogMiwgInJldHJvIjogWyJMYSBhcGFyaWVuY2lhIHZpc3VhbCBubyBkZWZpbmUgSDItUi4iLCAiSDItUiBxdWVkYXLDrWEgc2luIHRyYXphLiIsICJDb3JyZWN0bzogdW5hIG3DqXRyaWNhIGRldGVybWluYSBIMi1SIHkgb3RyYSBkZWNpc2nDs24gZ3XDrWEgbGEgZXhwbG9yYWNpw7NuLiIsICJObyBoYXkgcmF6w7NuIHBhcmEgZm9yemFyIGFtYmFzIGRlY2lzaW9uZXMuIl0sICJjb250ZXh0byI6ICJIMi1SIHVzYSBlbCBtw6F4aW1vOyBsYSBleHBsb3JhY2nDs24gcGVybWl0ZSBlbGVnaXIgZW50cmUgbG9zIHByaW5jaXBhbGVzIHByb3ZlZWRvcmVzLiIsICJ0b3RhbCI6IDEwfQ==")


### EJERCICIO S06-CONSULTA — escribe Cypher tú

Construye una consulta que cuente los **procesos distintos de la entidad ancla adjudicados al proveedor elegido**.

Requisitos:

- usa `$ancla`;
- usa `$proveedor`;
- devuelve exactamente una columna llamada `procesos`;
- cuenta `DISTINCT p`.

<details><summary><strong>Apoyo si te atascaste</strong></summary>

```cypher
MATCH (e:Entidad {nit:$ancla})-[:PUBLICA]->(p:Proceso)-[:ADJUDICADO_A]->(v:Proveedor {nit:$proveedor})
RETURN count(DISTINCT p) AS procesos
```
</details>


In [ ]:
consulta_propia = """
"""
if not consulta_propia.strip():
    raise ValueError("Escribe tu consulta Cypher; tienes un apoyo plegado encima.")

if modo_neo4j:
    respuesta = driver.execute_query(
        consulta_propia,
        ancla=nit_deseado,
        proveedor=str(proveedor_elegido["nit_proveedor"]),
    )
    if not respuesta.records:
        raise ValueError("Tu consulta no devolvió filas.")
    datos_respuesta = respuesta.records[0].data()
    if "procesos" not in datos_respuesta:
        raise ValueError("Tu consulta debe devolver exactamente una columna llamada procesos.")
    procesos_propios = int(datos_respuesta["procesos"])
else:
    procesos_propios = int(
        vecindario_df.loc[vecindario_df["nit_entidad"].eq(nit_deseado), "proceso"].nunique()
    )
    print("Consulta Cypher propia guardada pero PENDIENTE de ejecución en Aura.")

esperados_propios = int(proveedor_elegido["procesos_con_entidad"])
assert procesos_propios == esperados_propios, "Revisa DISTINCT, patrón y parámetros."
print("Consulta propia:", procesos_propios, "| esperado:", esperados_propios)


**Cómo se lee.** El número corresponde a procesos distintos para un par entidad–proveedor.

**Qué nos dice.** Tu consulta reproduce una métrica del contrato para una selección concreta.

**Qué NO permite concluir todavía.** Ese conteo no mide gasto ni calidad de la adjudicación.

**Qué error común.** Obtener el número correcto por casualidad con un patrón equivocado. Revisa también las relaciones y los parámetros.


### Antes del hito: registra dos decisiones

1. **Límite:** una conclusión que todavía no puedes sostener + el dato concreto que faltaría.
2. **Decisión de modelado:** por qué `Proceso` debe ser nodo y qué alternativa descartaste.

La evaluación no busca una frase genérica. Debe referirse a tu ejecución.


In [ ]:
#@title Autoevaluación 10 — Exportación auditable { display-mode: "form" }
pregunta_codificada("eyJudW1lcm8iOiAxMCwgInRlbWEiOiAiRXhwb3J0YWNpw7NuIGF1ZGl0YWJsZSIsICJwcmVndW50YSI6ICLCv1F1w6kgaGFjZSDDunRpbCBsYSBleHBvcnRhY2nDs24gcGFyYSBTNyB5IHBhcmEgdW5hIHJldmlzacOzbiBwb3N0ZXJpb3I/IiwgIm9wY2lvbmVzIjogWyJHdWFyZGFyIHNvbG8gdW5hIGltYWdlbiBkZWwgZ3JhZm8uIiwgIkd1YXJkYXIgSURzLCBkZXNjcmlwY2lvbmVzLCBVUkwsIHNlbGVjY2nDs24sIG3DqXRyaWNhcyB5IG1vdG9yIHJlYWwgZGUgZWplY3VjacOzbi4iLCAiR3VhcmRhciBsYSBjb250cmFzZcOxYSBkZSBBdXJhIGp1bnRvIGEgbGEgZmljaGEuIiwgIkV4cG9ydGFyIHRvZG8gU0VDT1Agc2luIGRlY2xhcmFyIGPDs21vIHNlIHNlbGVjY2lvbsOzLiJdLCAiY29ycmVjdGEiOiAxLCAicmV0cm8iOiBbIlVuYSBpbWFnZW4gYXl1ZGEgYSBjb211bmljYXIsIHBlcm8gbm8gY29uc2VydmEgdGV4dG8gbmkgdHJhemFiaWxpZGFkIHN1ZmljaWVudGUuIiwgIkNvcnJlY3RvOiBJRHMgKyB0ZXh0byArIHNlbGVjY2nDs24gKyBlc3RhZG8gZGVsIG1vdG9yIHBlcm1pdGVuIGNvbnRpbnVhciB5IGF1ZGl0YXIuIiwgIkxhcyBjcmVkZW5jaWFsZXMgbnVuY2Egc29uIGV2aWRlbmNpYSB5IG5vIGRlYmVuIGd1YXJkYXJzZS4iLCAiU2luIGNyaXRlcmlvIGRlIHNlbGVjY2nDs24gbm8gc2FiZW1vcyBxdcOpIHJlcHJlc2VudGEgZWwgYXJjaGl2by4iXSwgImNvbnRleHRvIjogIlM3IG5lY2VzaXRhIHRleHRvIGUgSURzIGRlIGxvcyBwcm9jZXNvcyBkZWwgdmVjaW5kYXJpbzsgbGEgZmljaGEgZGViZSBkZWNsYXJhciBxdcOpIHNlIGVqZWN1dMOzIHJlYWxtZW50ZS4iLCAidG90YWwiOiAxMH0=")


In [ ]:
if modo_neo4j:
    proveedor_h2r_neo = neo_df.iloc[0]
    assert str(proveedor_h2r_neo["nit_proveedor"]) == str(proveedor_h2r["nit_proveedor"])
    assert int(proveedor_h2r_neo["entidades_conectadas"]) == maximo_h2r

    if uso_respaldo_s06:
        desenlace_h2r_neo = "no evaluable con mi ancla; se ejecutó el respaldo en Neo4j"
    elif int(proveedor_h2r_neo["entidades_conectadas"]) > MEDIANA_H2R:
        desenlace_h2r_neo = "conexión más fuerte que la mediana de las candidatas de S5"
    else:
        desenlace_h2r_neo = "conexión igual o menor que la mediana de las candidatas de S5"
else:
    desenlace_h2r_neo = "PENDIENTE: no se ejecutó Neo4j"

print("Desenlace H2-R (Neo4j):", desenlace_h2r_neo)
print("Proveedor H2-R:", proveedor_h2r["nit_proveedor"], "| máximo:", maximo_h2r, "| mediana:", MEDIANA_H2R)
print("Proveedor explorado:", proveedor_elegido["nit_proveedor"], "| conexiones:", int(proveedor_elegido["entidades_conectadas"]))


In [ ]:
from pathlib import Path
from datetime import datetime, timezone

autor = input("Autor o alias de equipo (guardar solo en repositorio privado): ").strip()
decision_modelo = input("Justifica por qué Proceso debe ser nodo para tu pregunta: ").strip()
if not autor or len(decision_modelo) < 20:
    raise ValueError("Registra autor y una justificación de modelado suficientemente concreta.")

fecha_ejecucion = datetime.now(timezone.utc).isoformat()
limite_estudiante = input("Límite concreto y dato faltante: ").strip()
alternativa_modelo = input("Alternativa de modelado descartada: ").strip()
razon_alternativa = input("¿Por qué la descartaste para esta pregunta?: ").strip()

if len(limite_estudiante) < 25:
    raise ValueError("Nombra qué no puedes concluir y qué dato concreto faltaría.")
if len(alternativa_modelo) < 5 or len(razon_alternativa) < 15:
    raise ValueError("Nombra una alternativa real y explica por qué la descartaste.")


In [ ]:
#@title Guardar ficha y archivo para S7 { display-mode: "form" }
export = vecindario_df.merge(
    datos[["id_proceso", "descripcion", "modalidad", "url_secop"]].drop_duplicates("id_proceso"),
    left_on="proceso",
    right_on="id_proceso",
    how="left",
    validate="many_to_one",
)
assert export["id_proceso"].notna().all(), "Hay procesos sin correspondencia en el extracto."

export["nit_proveedor_explorado"] = str(proveedor_elegido["nit_proveedor"])
export["motor_ejecucion"] = "Neo4j" if modo_neo4j else "pandas; Neo4j pendiente"
export.to_json("s06_contexto_procesos.jsonl", orient="records", lines=True, force_ascii=False)

hito = f"""# Hito S06 — Ficha relacional de revisión

- Autor: {autor}
- Fecha UTC: {fecha_ejecucion}
- SHA256 del extracto: {huella_datos}
- Origen del ancla: {origen_ancla}
- Proceso de entrada: {ancla_original.get("id_proceso", "")}
- Proceso/entidad usados para el grafo: {ancla_trabajo.get("id_proceso", "")} — {ancla_trabajo.get("entidad", "")}
- Respaldo pedagógico: {uso_respaldo_s06}
- Motor: {"Neo4j" if modo_neo4j else "pandas; Neo4j pendiente"}
- Carga repetida: {carga_repetida if carga_repetida is not None else "PENDIENTE"}
- pandas == Neo4j: {coinciden if coinciden is not None else "PENDIENTE"}
- Proveedor que determina H2-R: {proveedor_h2r["nit_proveedor"]} — {proveedor_h2r["proveedor"]}
- Máximo H2-R: {maximo_h2r}
- Mediana H2-R: {MEDIANA_H2R}
- Desenlace H2-R pandas: {desenlace_h2r_pd}
- Desenlace H2-R Neo4j: {desenlace_h2r_neo}
- NIT proveedor explorado: {proveedor_elegido["nit_proveedor"]}
- Entidades conectadas proveedor explorado: {int(proveedor_elegido["entidades_conectadas"])}
- Otras entidades proveedor explorado: {otras_entidades}
- Procesos en el vecindario: {len(vecindario_df)}
- Decisión de exploración: {razon_exploracion}

## Límite
{limite_estudiante}

## Decisión de modelado
{decision_modelo}

### Alternativa descartada
{alternativa_modelo}

Razón: {razon_alternativa}

## Consulta propia (Cypher)
```cypher
{consulta_propia.strip()}
```
"""

Path("hito_s06_ficha_relacional.md").write_text(hito, encoding="utf-8")
print(hito)

try:
    from google.colab import files
    files.download("hito_s06_ficha_relacional.md")
    files.download("s06_contexto_procesos.jsonl")
except ImportError:
    print("Archivos generados en el runtime.")

print("Exportación S7:", len(export), "procesos; ficha y JSONL guardados.")


## Entrega — no te quedes a medio camino

1. En el repositorio **privado** de tu equipo abre o crea `hitos/s06/`.
2. Sube:
   - `hito_s06_ficha_relacional.md`
   - `s06_contexto_procesos.jsonl`
3. Haz el commit.
4. Abre el **commit**, no solo el archivo.
5. Pega la URL de ese commit en el aula.

### Antes de entregar comprueba

- ¿La ficha dice si el ancla fue propia o respaldo?
- ¿Diferencia proveedor H2-R y proveedor explorado?
- ¿Si usaste RESPALDO aparece `PENDIENTE` donde corresponde?
- ¿Tu consulta propia quedó guardada?
- ¿El límite dice qué dato falta?


---
# Hoja de trucos

```text
MATCH          busca un patrón
WHERE          filtra
WITH           pasa variables/resultados a la siguiente etapa
RETURN         define la salida
MERGE          encuentra o crea por identidad
SET            actualiza propiedades
UNWIND         convierte lista → filas
DISTINCT       evita duplicar unidades lógicas
ORDER BY       ordena
LIMIT          acota
DETACH DELETE  elimina nodo + relaciones
```

## Graph Lab en una línea

`RETURN count(*)` → tabla.  
`RETURN nodo, relacion, nodo` → grafo.  
`RETURN camino` → grafo del camino completo.

## Idea central

Cassandra preparó datos para una consulta repetitiva conocida. Neo4j hace de las **relaciones y los caminos** parte explícita de la pregunta.


## Lo que sigue

S6 ya construyó un vecindario relacional. Ahora ese vecindario contiene nombres, descripciones y URLs de procesos.

La nueva pregunta será:

> **¿Cuáles de esos procesos son textualmente más relevantes para una búsqueda concreta?**

`s06_contexto_procesos.jsonl` será la entrada de Elasticsearch/BM25 en S7.

**Recuerda:** el grafo ayudó a decidir *qué conjunto mirar*. La búsqueda textual ayudará a decidir *qué texto dentro de ese conjunto revisar primero*.


In [ ]:
if driver is not None:
    driver.close()
    print("Conexión Neo4j cerrada.")
else:
    print("Sin conexión Neo4j abierta (respaldo).")
